In [1]:
import pandas as pd

# =========================
# MASTER EVIDENCE FRAMEWORK
# =========================

evidence_sources = pd.DataFrame({

'study':[
    'UCAS',
    'ISUIA',
    'ISAT'
],

'purpose':[
    'rupture risk context',
    'treatment outcomes',
    'coiling vs clipping comparison'
],

'clinical_question':[
    'How likely is rupture?',
    'What are clipping outcomes?',
    'How do treatment modalities compare?'
]

})

display(evidence_sources)

,study,purpose,clinical_question
0,UCAS,rupture risk context,How likely is rupture?
1,ISUIA,treatment outcomes,What are clipping outcomes?
2,ISAT,coiling vs clipping comparison,How do treatment modalities compare?


In [2]:
# =========================
# 1. Imports
# =========================
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


# =========================
# 2. Optional: show available files
# =========================
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# =========================
# 3. Load datasets
# =========================
clinical_df = pd.read_csv(
    '/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/clinical_all.csv'
)

morph_df = pd.read_csv(
    '/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/morphological_aneurysm_artery.csv'
)


# =========================
# 4. Merge on patient ID
# =========================
df = pd.merge(clinical_df, morph_df, on='number')

# Keep only rows where PHASE exists
df = df.dropna(subset=['PHASE'])


# =========================
# 5. Define input features and target
# =========================
clinical_features = [
    # Clinical features
    'Age',
    'Systolic Pressure',
    'Diastolic Pressure',
    'Diabetes history',
    'Hypertension history',
    'location',

    # Morphological / aneurysm features
    'Neck width (NW)',
    'Aneurysm perpendicular height (H)',
    'Maximum length of the aneurysm(L_1)',
    'Maximum width of the aneurysm (L_2)',
    'D_max',
    'BottleneckFactor',
    'ConicityParameter',
    'Aneurysm volume (V)',
    'EllipticityIndex',
    'NonSphericityIndex',
    'UndulationIndex',
    ' The ratio of H to NW(AR)',
    ' The ratio of L1 to L2(LR)',
    'The ratio of H to the average of D1, D2, and D3.(SR)'
]

X = df[clinical_features].copy()
y = df['PHASE']


# =========================
# 6. Encode categorical columns
# =========================
categorical_cols = [
    'Diabetes history',
    'Hypertension history',
    'location'
]
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le


# =========================
# 7. Fill missing numeric values
# =========================
X = X.fillna(X.median(numeric_only=True))


# =========================
# 8. Train/test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# =========================
# 9. Train model
# =========================
model = RandomForestRegressor(
    n_estimators=600,
    max_depth=12,
    min_samples_split=3,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42
)

model.fit(X_train, y_train)


# =========================
# 10. Predict and evaluate
# =========================
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'PHASE - MSE: {mse:.2f}, R2: {r2:.2f}')


# =========================
# 11. Feature importance
# =========================
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)


# =========================
# 12. Decision-support post-processing layer
# =========================
# Prototype thresholds based on current model output distribution
# These are for routing/testing the evidence layer, not final clinical cutoffs.

def phase_to_category(score):
    if score <= 1.75:
        return "Low Risk"
    elif score <= 2.3:
        return "Moderate Risk"
    else:
        return "High Risk"

def physician_recommendation(score):
    if score <= 1.75:
        return "Leans toward conservative management; monitor with interval imaging."
    elif score <= 2.3:
        return "Intermediate risk; treatment decision should consider morphology, age, rupture/growth risk, and outcome data from similar cases."
    else:
        return "Higher-risk profile; intervention should be discussed, with treatment modality individualized."

def patient_summary(score):
    if score <= 1.75:
        return "Your aneurysm appears lower risk and may be appropriate for monitoring."
    elif score <= 2.3:
        return "Your aneurysm appears to have an intermediate level of risk, so monitoring and treatment options should be discussed carefully."
    else:
        return "Your aneurysm appears higher risk, so treatment may be discussed as an option."

def evidence_stub(score):
    if score <= 1.75:
        return "Would link to literature and institutional outcomes for surveillance in lower-risk aneurysms."
    elif score <= 2.3:
        return "Would link to literature comparing surveillance versus intervention in intermediate-risk aneurysms."
    else:
        return "Would link to literature and institutional outcomes for treatment of higher-risk aneurysms."

# =========================
# 13. Build results table
# =========================
results_df = X_test.copy()
results_df["Actual_PHASE"] = y_test.values
results_df["Predicted_PHASE"] = y_pred
results_df["Predicted_PHASE_Rounded"] = results_df["Predicted_PHASE"].round(1)

results_df["Risk_Category"] = results_df["Predicted_PHASE"].apply(phase_to_category)
results_df["Physician_Recommendation"] = results_df["Predicted_PHASE"].apply(physician_recommendation)
results_df["Patient_Summary"] = results_df["Predicted_PHASE"].apply(patient_summary)
results_df["Evidence_Mapping"] = results_df["Predicted_PHASE"].apply(evidence_stub)


# =========================
# 14. Show compact output
# =========================
display_cols = [
    "Actual_PHASE",
    "Predicted_PHASE_Rounded",
    "Risk_Category",
    "Physician_Recommendation",
    "Patient_Summary",
    "Evidence_Mapping"
]

results_df[display_cols].head(10)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importance)


## PHASES score prediction pipeline.
## A machine learning model (Random Forest) is used to predict PHASES scores from clinical and morphological features.
## Feature importance is computed to identify which variables contribute most to the prediction.

/kaggle/input/datasets/eshaanisingh/ucas-table-2-rupture-risk-by-size-and-location/DR App Literature Data - UCAS Japan_ .csv
/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/morphological_control.csv
/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/hemodynamic_aneurysm_artery.csv
/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/morphological_aneurysm_artery.csv
/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/clinical_all.csv
/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/hemodynamic_control.csv
/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/controls/AHMU1218138/3D_healthy_artery_AHMU1218138.stl
/kaggle/input/datasets/eshaanisingh/intracranial-aneurysm-dataset/DR APP Public Dataset/controls/AHMU1218138/cta_images_head_AHMU1218138.nii
/kaggle/input/data

In [3]:
# ==========================================
# UCAS ANATOMICAL LOCATION MAPPING
# ==========================================

# Recover the original anatomical location
# before LabelEncoder converted it to numbers.
results_df["Original_Location"] = df.loc[X_test.index, "location"].values

# Map the dataset's MCA-specific labels to the
# anatomical category used by UCAS.
location_map = {
    "M1": "Middle cerebral artery",
    "M1-M2 junction": "Middle cerebral artery",
    "M2": "Middle cerebral artery",
    "M3": "Middle cerebral artery"
}

results_df["UCAS_Location"] = (
    results_df["Original_Location"].map(location_map)
)

print("Original dataset locations:")
print(results_df["Original_Location"].value_counts())

print("\nMapped UCAS locations:")
print(results_df["UCAS_Location"].value_counts(dropna=False))

Original dataset locations:
Original_Location
M1                10
M2                 6
M1-M2 junction     3
Name: count, dtype: int64

Mapped UCAS locations:
UCAS_Location
Middle cerebral artery    19
Name: count, dtype: int64


In [4]:
print(results_df.columns.tolist())

['Age', 'Systolic Pressure', 'Diastolic Pressure', 'Diabetes history', 'Hypertension history', 'location', 'Neck width (NW)', 'Aneurysm perpendicular height (H)', 'Maximum length of the aneurysm(L_1)', 'Maximum width of the aneurysm (L_2)', 'D_max', 'BottleneckFactor', 'ConicityParameter', 'Aneurysm volume (V)', 'EllipticityIndex', 'NonSphericityIndex', 'UndulationIndex', ' The ratio of H to NW(AR)', ' The ratio of L1 to L2(LR)', 'The ratio of H to the average of D1, D2, and D3.(SR)', 'Actual_PHASE', 'Predicted_PHASE', 'Predicted_PHASE_Rounded', 'Risk_Category', 'Physician_Recommendation', 'Patient_Summary', 'Evidence_Mapping', 'Original_Location', 'UCAS_Location']


In [5]:
# =========================
# 15. Create Evidence Table
# =========================

import pandas as pd

evidence_df = pd.DataFrame([
    {
        "study": "ISUIA",
        "rupture_status": "unruptured",
        "phase_group": "low",
        "location_group": "anterior",
        "treatment_context": "conservative",
        "summary": "Low rupture risk observed in small anterior circulation aneurysms; monitoring often appropriate."
    },
    {
        "study": "UCAS Japan",
        "rupture_status": "unruptured",
        "phase_group": "moderate_or_high",
        "location_group": "high-risk",
        "treatment_context": "consider treatment",
        "summary": "Higher rupture risk in aneurysms with larger size or high-risk locations such as ACom or posterior circulation."
    },
    {
        "study": "ISAT",
        "rupture_status": "ruptured",
        "phase_group": "any",
        "location_group": "any",
        "treatment_context": "intervention",
        "summary": "Coiling and clipping outcomes differ; coiling often associated with better short-term outcomes in selected patients."
    }
])

print(evidence_df)

## Evidence abstraction layer:
## This component is designed to map clinical scenarios (e.g., risk category, patient history)
## to relevant literature-derived outcomes.
## Currently, this layer uses structured data extracted from ISUIA (Table 4) as a proof-of-concept.
## This will eventually include additional landmark studies and more outcome data.

        study rupture_status       phase_group location_group  \
0       ISUIA     unruptured               low       anterior   
1  UCAS Japan     unruptured  moderate_or_high      high-risk   
2        ISAT       ruptured               any            any   

    treatment_context                                            summary  
0        conservative  Low rupture risk observed in small anterior ci...  
1  consider treatment  Higher rupture risk in aneurysms with larger s...  
2        intervention  Coiling and clipping outcomes differ; coiling ...  


In [6]:
import pandas as pd

evidence_df = pd.read_csv('/kaggle/input/datasets/eshaanisingh/isuia-intervention-and-outcomes/DR App Literature Data - Sheet1 (1).csv')

evidence_df.head()

print(evidence_df.columns)

def get_clipping_outcomes(evidence_df):
    return evidence_df[evidence_df['intervention'] == 'clipping']

clipping_data = get_clipping_outcomes(evidence_df)
clipping_data.head()

# Example: always show clipping outcomes for now
clipping_summary = evidence_df[
    (evidence_df['intervention'] == 'clipping') &
    (evidence_df['timepoint'] == '1 year') &
    (evidence_df['outcome_category'] == 'overall_morbidity_mortality') &
    (evidence_df['outcome_subcategory'] == 'all patients')
]

print(clipping_summary)

## Extracted and displaying structured outcome data from Table 4 of the ISUIA study.
## This table specifically contains 1-year morbidity and mortality outcomes following surgical clipping,
## stratified by patient groups.

Index(['table_number', 'group_label', 'group_n', 'intervention', 'timepoint',
       'outcome_category', 'outcome_subcategory', 'percent', 'ci_lower',
       'ci_upper'],
      dtype='object')
   table_number group_label  group_n intervention timepoint  \
24      Table 4     Group 1      798     clipping    1 year   
25      Table 4     Group 2      197     clipping    1 year   

               outcome_category outcome_subcategory  percent  ci_lower  \
24  overall_morbidity_mortality        all patients     15.7      13.2   
25  overall_morbidity_mortality        all patients     13.1       8.0   

    ci_upper  
24      18.2  
25      17.2  


In [7]:
import pandas as pd


evidence_df = pd.read_csv('/kaggle/input/datasets/eshaanisingh/isuia-intervention-and-outcomes/DR App Literature Data - Sheet1 (1).csv')




# STEP 1: Filter the correct row from evidence dataset
clipping_1yr_all = evidence_df[
    (evidence_df["intervention"] == "clipping") &
    (evidence_df["timepoint"] == "1 year") &
    (evidence_df["outcome_category"] == "overall_morbidity_mortality") &
    (evidence_df["outcome_subcategory"] == "all patients")
]

# STEP 2: Convert to summary text
if len(clipping_1yr_all) > 0:
    clipping_percent = clipping_1yr_all.iloc[0]["percent"]
    clipping_summary_text = f"ISUIA clipping data: 1-year overall morbidity/mortality was {clipping_percent}."
else:
    clipping_summary_text = "No matching clipping outcome found."

print(clipping_summary_text)



# =========================
# STEP 3: ATTACH ISUIA OUTPUT
# =========================

# FIX: ensure function name matches what we apply
def intervention_outcome_summary(row):

    if row["Risk_Category"] in ["Moderate Risk", "High Risk"]:

        group = row["ISUIA_Group"]

        subset = evidence_df[
            (evidence_df["intervention"] == "clipping") &
            (evidence_df["timepoint"] == "1 year") &
            (evidence_df["outcome_category"] == "overall_morbidity_mortality") &
            (evidence_df["group_label"] == group)
        ]

        if len(subset) > 0:

            percent = subset.iloc[0]["percent"]
            lower = subset.iloc[0]["ci_lower"]
            upper = subset.iloc[0]["ci_upper"]

            return (
                f"{group}: 1-year morbidity/mortality = "
                f"{percent}% (CI {lower}–{upper}%)"
            )

        return "No matching ISUIA data found."

    return "Low-risk case; surveillance typically recommended."


# No prior SAH variable available in current dataset.
# Default all cases to Group 1 for ISUIA lookup.

results_df["ISUIA_Group"] = "Group 1"

# APPLY correct function
results_df["Intervention_Outcome_Summary"] = results_df.apply(
    intervention_outcome_summary,
    axis=1
)

# =========================
# STEP 4: Display final table
# =========================

display_cols = [
    "Actual_PHASE",
    "Predicted_PHASE_Rounded",
    "Risk_Category",
    "Physician_Recommendation",
    "Intervention_Outcome_Summary",
    "Patient_Summary"
]

display(results_df[display_cols].reset_index(drop=True).head(10))



ISUIA clipping data: 1-year overall morbidity/mortality was 15.7.


,Actual_PHASE,Predicted_PHASE_Rounded,Risk_Category,Physician_Recommendation,Intervention_Outcome_Summary,Patient_Summary
0,3.0,1.8,Moderate Risk,Intermediate risk; treatment decision should c...,Group 1: 1-year morbidity/mortality = 15.7% (C...,Your aneurysm appears to have an intermediate ...
1,3.0,2.1,Moderate Risk,Intermediate risk; treatment decision should c...,Group 1: 1-year morbidity/mortality = 15.7% (C...,Your aneurysm appears to have an intermediate ...
2,2.0,2.0,Moderate Risk,Intermediate risk; treatment decision should c...,Group 1: 1-year morbidity/mortality = 15.7% (C...,Your aneurysm appears to have an intermediate ...
3,2.0,2.5,High Risk,Higher-risk profile; intervention should be di...,Group 1: 1-year morbidity/mortality = 15.7% (C...,"Your aneurysm appears higher risk, so treatmen..."
4,1.0,1.8,Moderate Risk,Intermediate risk; treatment decision should c...,Group 1: 1-year morbidity/mortality = 15.7% (C...,Your aneurysm appears to have an intermediate ...
5,1.0,1.6,Low Risk,Leans toward conservative management; monitor ...,Low-risk case; surveillance typically recommen...,Your aneurysm appears lower risk and may be ap...
6,0.0,1.4,Low Risk,Leans toward conservative management; monitor ...,Low-risk case; surveillance typically recommen...,Your aneurysm appears lower risk and may be ap...
7,2.0,2.0,Moderate Risk,Intermediate risk; treatment decision should c...,Group 1: 1-year morbidity/mortality = 15.7% (C...,Your aneurysm appears to have an intermediate ...
8,3.0,2.5,High Risk,Higher-risk profile; intervention should be di...,Group 1: 1-year morbidity/mortality = 15.7% (C...,"Your aneurysm appears higher risk, so treatmen..."
9,2.0,1.4,Low Risk,Leans toward conservative management; monitor ...,Low-risk case; surveillance typically recommen...,Your aneurysm appears lower risk and may be ap...


In [8]:
print(results_df.columns.tolist())

['Age', 'Systolic Pressure', 'Diastolic Pressure', 'Diabetes history', 'Hypertension history', 'location', 'Neck width (NW)', 'Aneurysm perpendicular height (H)', 'Maximum length of the aneurysm(L_1)', 'Maximum width of the aneurysm (L_2)', 'D_max', 'BottleneckFactor', 'ConicityParameter', 'Aneurysm volume (V)', 'EllipticityIndex', 'NonSphericityIndex', 'UndulationIndex', ' The ratio of H to NW(AR)', ' The ratio of L1 to L2(LR)', 'The ratio of H to the average of D1, D2, and D3.(SR)', 'Actual_PHASE', 'Predicted_PHASE', 'Predicted_PHASE_Rounded', 'Risk_Category', 'Physician_Recommendation', 'Patient_Summary', 'Evidence_Mapping', 'Original_Location', 'UCAS_Location', 'ISUIA_Group', 'Intervention_Outcome_Summary']


In [9]:
##UCAS Evidence Layer

ucas_df = pd.read_csv('/kaggle/input/datasets/eshaanisingh/ucas-table-2-rupture-risk-by-size-and-location/DR App Literature Data - UCAS Japan_ .csv')

print(ucas_df.columns)

display(ucas_df.head())




Index(['Location of Aneurysm', '3–4 mm', '5–6 mm', '7–9 mm', '10–24 mm',
       '≥25 mm'],
      dtype='object')


,Location of Aneurysm,3–4 mm,5–6 mm,7–9 mm,10–24 mm,≥25 mm
0,Middle cerebral artery,0.23 (0.09–0.54),0.31 (0.10–0.96),1.56 (0.74–3.26),4.11 (2.22–7.66),16.87 (2.38–119.77)
1,Anterior communicating artery,0.90 (0.45–1.80),0.75 (0.28–2.02),1.97 (0.82–4.76),5.24 (1.97–13.95),39.77 (9.95–159.00)
2,Internal carotid artery,0.14 (0.04–0.57),0,1.19 (0.30–4.77),1.07 (0.27–4.28),10.61 (1.49–75.3)
3,Internal carotid–posterior communicating artery,0.41 (0.15–1.10),1.00 (0.37–2.66),3.19 (1.66–6.12),6.12 (1.66–6.13),126.97 (40.95–393.68)
4,Basilar tip and basilar–superior cerebellar ar...,0.23 (0.03–1.61),0.46 (0.06–3.27),0.97 (0.24–3.89),6.94 (3.74–12.90),117.82 (16.60–836.43)


In [10]:
# =========================
# REFORMAT UCAS TABLE
# =========================

ucas_long = ucas_df.melt(
    id_vars="Location of Aneurysm",
    var_name="size_category",
    value_name="rupture_data"
)

display(ucas_long.head())

,Location of Aneurysm,size_category,rupture_data
0,Middle cerebral artery,3–4 mm,0.23 (0.09–0.54)
1,Anterior communicating artery,3–4 mm,0.90 (0.45–1.80)
2,Internal carotid artery,3–4 mm,0.14 (0.04–0.57)
3,Internal carotid–posterior communicating artery,3–4 mm,0.41 (0.15–1.10)
4,Basilar tip and basilar–superior cerebellar ar...,3–4 mm,0.23 (0.03–1.61)


In [11]:
# =========================
# SPLIT UCAS RUPTURE DATA
# =========================

import re

def split_ucas_values(value):

    value = str(value).strip()

    # Handle zero values
    if value == '0':
        return pd.Series([0.0, None, None])

    pattern = r'([\d\.]+)\s*\(([\d\.]+)[–-]([\d\.]+)\)'

    match = re.match(pattern, value)

    if match:

        annual_rate = float(match.group(1))
        ci_lower = float(match.group(2))
        ci_upper = float(match.group(3))

        return pd.Series([annual_rate, ci_lower, ci_upper])

    return pd.Series([None, None, None])

ucas_long[['annual_rupture_rate',
           'ci_lower',
           'ci_upper']] = ucas_long['rupture_data'].apply(split_ucas_values)

display(ucas_long.head())

,Location of Aneurysm,size_category,rupture_data,annual_rupture_rate,ci_lower,ci_upper
0,Middle cerebral artery,3–4 mm,0.23 (0.09–0.54),0.23,0.09,0.54
1,Anterior communicating artery,3–4 mm,0.90 (0.45–1.80),0.90,0.45,1.80
2,Internal carotid artery,3–4 mm,0.14 (0.04–0.57),0.14,0.04,0.57
3,Internal carotid–posterior communicating artery,3–4 mm,0.41 (0.15–1.10),0.41,0.15,1.10
4,Basilar tip and basilar–superior cerebellar ar...,3–4 mm,0.23 (0.03–1.61),0.23,0.03,1.61


In [12]:
results_df["Risk_Category"].value_counts()

Risk_Category
Low Risk         7
Moderate Risk    6
High Risk        6
Name: count, dtype: int64

In [13]:
results_df[
    ["Predicted_PHASE_Rounded", "Risk_Category", "Intervention_Outcome_Summary"]
].head(10)

,Predicted_PHASE_Rounded,Risk_Category,Intervention_Outcome_Summary
40,1.8,Moderate Risk,Group 1: 1-year morbidity/mortality = 15.7% (C...
22,2.1,Moderate Risk,Group 1: 1-year morbidity/mortality = 15.7% (C...
55,2.0,Moderate Risk,Group 1: 1-year morbidity/mortality = 15.7% (C...
72,2.5,High Risk,Group 1: 1-year morbidity/mortality = 15.7% (C...
0,1.8,Moderate Risk,Group 1: 1-year morbidity/mortality = 15.7% (C...
26,1.6,Low Risk,Low-risk case; surveillance typically recommen...
39,1.4,Low Risk,Low-risk case; surveillance typically recommen...
67,2.0,Moderate Risk,Group 1: 1-year morbidity/mortality = 15.7% (C...
10,2.5,High Risk,Group 1: 1-year morbidity/mortality = 15.7% (C...
44,1.4,Low Risk,Low-risk case; surveillance typically recommen...


In [14]:
# =========================
# UCAS SIZE CATEGORY
# =========================

def ucas_size_category(dmax):

    if pd.isna(dmax):
        return None

    if 3 <= dmax < 5:
        return "3–4 mm"

    elif 5 <= dmax < 7:
        return "5–6 mm"

    elif 7 <= dmax < 10:
        return "7–9 mm"

    elif 10 <= dmax < 25:
        return "10–24 mm"

    elif dmax >= 25:
        return "≥25 mm"

    else:
        return "<3 mm"

results_df["UCAS_Size_Category"] = results_df["D_max"].apply(ucas_size_category)

results_df[["D_max", "UCAS_Size_Category"]].head(10)

,D_max,UCAS_Size_Category
40,3.770882,3–4 mm
22,4.814564,3–4 mm
55,6.873307,5–6 mm
72,6.522473,5–6 mm
0,3.203880,3–4 mm
26,3.606101,3–4 mm
39,4.379373,3–4 mm
67,9.169086,7–9 mm
10,7.075463,7–9 mm
44,4.373083,3–4 mm


In [15]:
# =========================
# UCAS LOOKUP FUNCTION
# =========================

def get_ucas_evidence(row):

    subset = ucas_long[
        (ucas_long["Location of Aneurysm"] == "Middle cerebral artery") &
        (ucas_long["size_category"] == row["UCAS_Size_Category"])
    ]

    if subset.empty:
        return "No matching UCAS evidence found."

    evidence = subset.iloc[0]

    return (
        f"UCAS: Annual rupture risk = "
        f"{evidence['annual_rupture_rate']}% "
        f"(95% CI {evidence['ci_lower']}–{evidence['ci_upper']}%)"
    )


##

results_df["UCAS_Rupture_Context"] = results_df.apply(
    get_ucas_evidence,
    axis=1
)

results_df[
    [
        "D_max",
        "UCAS_Size_Category",
        "UCAS_Rupture_Context"
    ]
].head(10)

,D_max,UCAS_Size_Category,UCAS_Rupture_Context
40,3.770882,3–4 mm,UCAS: Annual rupture risk = 0.23% (95% CI 0.09...
22,4.814564,3–4 mm,UCAS: Annual rupture risk = 0.23% (95% CI 0.09...
55,6.873307,5–6 mm,UCAS: Annual rupture risk = 0.31% (95% CI 0.1–...
72,6.522473,5–6 mm,UCAS: Annual rupture risk = 0.31% (95% CI 0.1–...
0,3.203880,3–4 mm,UCAS: Annual rupture risk = 0.23% (95% CI 0.09...
26,3.606101,3–4 mm,UCAS: Annual rupture risk = 0.23% (95% CI 0.09...
39,4.379373,3–4 mm,UCAS: Annual rupture risk = 0.23% (95% CI 0.09...
67,9.169086,7–9 mm,UCAS: Annual rupture risk = 1.56% (95% CI 0.74...
10,7.075463,7–9 mm,UCAS: Annual rupture risk = 1.56% (95% CI 0.74...
44,4.373083,3–4 mm,UCAS: Annual rupture risk = 0.23% (95% CI 0.09...


In [16]:
# =====================================
# CLINICAL REPORT GENERATOR
# =====================================

def generate_patient_report(patient_index):

    patient = results_df.iloc[patient_index]

    print("="*60)
    print("CLINICAL DECISION SUPPORT REPORT")
    print("="*60)



    print("\nPATIENT CHARACTERISTICS")
    print("----------------------")

    print(f"Age: {patient['Age']} years")

    print(f"Systolic Blood Pressure: {patient['Systolic Pressure']} mmHg")

    print(f"Diastolic Blood Pressure: {patient['Diastolic Pressure']} mmHg")

    print(f"Maximum Diameter (Dmax): {patient['D_max']:.2f} mm")

    print(f"Aspect Ratio: {patient[' The ratio of H to NW(AR)']:.2f}")

    print(f"Hypertension History: {patient['Hypertension history']}")

    print(f"Diabetes History: {patient['Diabetes history']}")


    
    print("\nPREDICTED PHASES SCORE")
    print("----------------------")
    print(patient["Predicted_PHASE_Rounded"])

    print("\nRISK CATEGORY")
    print("-------------")
    print(patient["Risk_Category"])

    print("\nNATURAL HISTORY (UCAS)")
    print("----------------------")
    print(patient["UCAS_Rupture_Context"])

    print("\nTREATMENT OUTCOMES (ISUIA)")
    print("--------------------------")
    print(patient["Intervention_Outcome_Summary"])

    print("\nPHYSICIAN RECOMMENDATION")
    print("------------------------")
    print(patient["Physician_Recommendation"])

    print("\nPATIENT SUMMARY")
    print("----------------")
    print(patient["Patient_Summary"])

    print("\nDISCLAIMER")
    print("----------")
    print("This prototype is intended to support—not replace—clinical judgment. Literature outputs are retrieved from published studies and should be interpreted in the context of individual patient factors.")

generate_patient_report(0)




CLINICAL DECISION SUPPORT REPORT

PATIENT CHARACTERISTICS
----------------------
Age: 71 years
Systolic Blood Pressure: 150 mmHg
Diastolic Blood Pressure: 96 mmHg
Maximum Diameter (Dmax): 3.77 mm
Aspect Ratio: 0.64
Hypertension History: 1
Diabetes History: 0

PREDICTED PHASES SCORE
----------------------
1.8

RISK CATEGORY
-------------
Moderate Risk

NATURAL HISTORY (UCAS)
----------------------
UCAS: Annual rupture risk = 0.23% (95% CI 0.09–0.54%)

TREATMENT OUTCOMES (ISUIA)
--------------------------
Group 1: 1-year morbidity/mortality = 15.7% (CI 13.2–18.2%)

PHYSICIAN RECOMMENDATION
------------------------
Intermediate risk; treatment decision should consider morphology, age, rupture/growth risk, and outcome data from similar cases.

PATIENT SUMMARY
----------------
Your aneurysm appears to have an intermediate level of risk, so monitoring and treatment options should be discussed carefully.

DISCLAIMER
----------
This prototype is intended to support—not replace—clinical judgmen